# 09 — Retrieval Evaluation

Compute **Hit Rate@K**, **MRR**, and **nDCG@K** on a synthetic labelled set.

**Ground truth construction:** For each query, take the **top-1 row ID** returned by similarity search as the target. Then score every other retriever against that target.

**Config source:** `configs/default.yaml` → `notebooks.retrieval_queries`

In [ ]:
import math
from rag_pipeline.utils import load_notebook_config
from rag_pipeline.embeddings import build_embeddings
from rag_pipeline.vectorstores import load_vectorstore
from rag_pipeline.retrieval import build_retriever

cfg, REPO = load_notebook_config()
FAISS_DIR = REPO / cfg.paths["faiss_index"]
QUERIES   = cfg.notebooks["retrieval_queries"]

emb = build_embeddings(dict(cfg.embeddings))
store = load_vectorstore(emb, {"type": "faiss", "persist_dir": str(FAISS_DIR)})

**Build ground truth**

In [ ]:
baseline = build_retriever(store, {"search_type": "similarity", "k": 1})

ground_truth = {
    q: baseline.invoke(q)[0].metadata.get("row")
    for q in QUERIES
}

print("Ground truth:")
for q, row in ground_truth.items():
    print(f"  row={row:>4} | {q[:60]}...")

**Metric functions**

In [ ]:
def hit_rate_at_k(results, gt, k):
    return sum(
        any(d.metadata.get("row") == gt[q] for d in docs[:k])
        for q, docs in results.items()
    ) / len(gt)


def mrr(results, gt):
    rr = 0.0
    for q, docs in results.items():
        for rank, d in enumerate(docs, start=1):
            if d.metadata.get("row") == gt[q]:
                rr += 1.0 / rank
                break
    return rr / len(gt)


def ndcg_at_k(results, gt, k):
    scores = []
    for q, docs in results.items():
        dcg = sum(
            1.0 / math.log2(rank + 1)
            for rank, d in enumerate(docs[:k], start=1)
            if d.metadata.get("row") == gt[q]
        )
        scores.append(dcg)          # IDCG = 1.0 for single relevant item
    return sum(scores) / len(scores)

**Compare retrievers**

In [ ]:
CONFIGS = {
    "similarity k=1":     {"search_type": "similarity", "k": 1},
    "similarity k=3":     {"search_type": "similarity", "k": 3},
    "similarity k=5":     {"search_type": "similarity", "k": 5},
    "mmr k=3 λ=0.5":      {"search_type": "mmr", "k": 3, "fetch_k": 20, "lambda_mult": 0.5},
    "mmr k=3 λ=0.2":      {"search_type": "mmr", "k": 3, "fetch_k": 20, "lambda_mult": 0.2},
    "threshold 0.4 k=5":  {"search_type": "similarity_score_threshold", "k": 5, "score_threshold": 0.4},
}

print(f"{'Configuration':<22} {'Hit@3':>8} {'MRR':>8} {'nDCG@5':>8}")
print("-" * 50)

for name, rcfg in CONFIGS.items():
    r = build_retriever(store, rcfg)
    results = {q: r.invoke(q) for q in QUERIES}
    print(f"{name:<22} {hit_rate_at_k(results, ground_truth, 3):>8.3f} "
          f"{mrr(results, ground_truth):>8.3f} "
          f"{ndcg_at_k(results, ground_truth, 5):>8.3f}")